In [1]:
from datetime import datetime

from pydantic import BaseModel, field_validator, ValidationError

In [2]:
class Model(BaseModel):
    dt: datetime

In [3]:
Model(dt="2020-01-01T12:00:00")

Model(dt=datetime.datetime(2020, 1, 1, 12, 0))

In [4]:
Model(dt="2020-01-01T12:00:00Z")

Model(dt=datetime.datetime(2020, 1, 1, 12, 0, tzinfo=TzInfo(0)))

In [5]:
try:
    Model(dt="2020/1/1 3:00pm")
except ValidationError as ex:
    print(ex)

1 validation error for Model
dt
  Input should be a valid datetime or date, invalid date separator, expected `-` [type=datetime_from_date_parsing, input_value='2020/1/1 3:00pm', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/datetime_from_date_parsing


In [6]:
try:
    Model(dt="Jan 1, 2020 3:00pm")
except ValidationError as ex:
    print(ex)

1 validation error for Model
dt
  Input should be a valid datetime or date, invalid character in year [type=datetime_from_date_parsing, input_value='Jan 1, 2020 3:00pm', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/datetime_from_date_parsing


In [7]:
from dateutil.parser import parse

In [8]:
parse("2020/1/1 3pm")

datetime.datetime(2020, 1, 1, 15, 0)

In [9]:
parse("Jan 1, 2020 3pm")

datetime.datetime(2020, 1, 1, 15, 0)

In [10]:
try:
    parse(datetime(2020, 1, 1, 15, 0, 0))
except TypeError as ex:
    print(ex)

Parser must be a string or character stream, not datetime


In [11]:
from typing import Any


class Model(BaseModel):
    dt: datetime

    @field_validator("dt", mode="before")
    @classmethod
    def parse_datetime(cls, value: Any):
        if isinstance(value, str):
            print("parsing string")
            try:
                return parse(value)
            except Exception as ex:
                raise ValueError(str(ex))
        print("pass through...")
        return value

In [12]:
Model(dt="2020/1/1 3pm")

parsing string


Model(dt=datetime.datetime(2020, 1, 1, 15, 0))

In [13]:
Model(dt=datetime(2020, 1, 1))

pass through...


Model(dt=datetime.datetime(2020, 1, 1, 0, 0))

In [14]:
try:
    Model(dt=[1, 2, 3])
except ValidationError as ex:
    print(ex)

pass through...
1 validation error for Model
dt
  Input should be a valid datetime [type=datetime_type, input_value=[1, 2, 3], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/datetime_type


In [15]:
class Model(BaseModel):
    number: int

    @field_validator("number", mode="before")
    @classmethod
    def validator_1(cls, value):
        print("running validator_1")
        return value

    @field_validator("number", mode="before")
    @classmethod
    def validator_2(cls, value):
        print("running validator_2")
        return value

    @field_validator("number", mode="before")
    @classmethod
    def validator_3(cls, value):
        print("running validator_3")
        return value


In [16]:
Model(number=1)

running validator_3
running validator_2
running validator_1


Model(number=1)